# Exploración de memoria conversacional (LangGraph) y *tool calling*
---

**Autor:** Jaime Lozano Cillero

# 0. Preparación del entorno

In [1]:
from dataclasses import dataclass, field
from datetime import datetime
import json
import logging
import time
import uuid
import requests
from typing import Any, Literal

from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    trim_messages,
)
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import tools_condition
from pydantic import BaseModel, Field

import getpass
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-nano")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

PDF_PATH = Path("../../data/raw/TENERIFE.pdf")
RAG_K = 5

MAX_TOOL_RETRIES = 3

# Retraso en segundos
STREAMING_DELAY_PER_CHAR = 0.003

print("Modelo generativo:", GENERATION_MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("PDF:", PDF_PATH.resolve())

Modelo generativo: gpt-4.1-nano
Modelo de embeddings: text-embedding-3-small
PDF: C:\Users\Jaime\Documents\Pontia\modulo-06-llms\entrega-final\pontia-llms-entrega-final\data\raw\TENERIFE.pdf


## 0.1 Observabilidad y Logging de Herramientas

Persistencia a nivel de sesión/notebook. Permite registrar ejecuciones, recibir callbacks en tiempo real y mostrar tablas o resúmenes de intentos.

In [2]:
from dataclasses import dataclass, field
from datetime import datetime
import json
import logging
import time
from typing import Any

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage,
    trim_messages,
)
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import tools_condition
from pydantic import BaseModel, Field


@dataclass
class ToolExecution:
    """Registro observable de una llamada real a herramienta con soporte de reintentos."""

    name: str
    arguments: dict[str, Any]
    ok: bool
    output: Any
    elapsed_seconds: float
    attempt: int = 1
    max_retries: int = 1
    timestamp: datetime = field(default_factory=datetime.now)

    def to_report(self) -> str:
        """Devuelve un informe legible del intento individual."""
        status_label = "✅ OK" if self.ok else "❌ ERROR"
        formatted_args = json.dumps(self.arguments, ensure_ascii=False)
        formatted_output = (
            json.dumps(self.output, ensure_ascii=False, indent=2)
            if isinstance(self.output, (dict, list))
            else str(self.output)
        )

        return (
            f"┌─ [TOOL EXECUTION] {self.name} ({status_label}) ────────────────────────\n"
            f"│ Timestamp : {self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}\n"
            f"│ Intento   : {self.attempt}/{self.max_retries}\n"
            f"│ Duración  : {self.elapsed_seconds:.4f}s\n"
            f"│ Argumentos: {formatted_args}\n"
            f"│ Resultado :\n"
            + "\n".join(f"│   {line}" for line in formatted_output.splitlines())
            + "\n└─────────────────────────────────────────────────────────────"
        )


class ToolExecutionLogger:
    """Buffer de auditoría que registra cada intento individual de ejecución."""

    def __init__(self, verbose: bool = True) -> None:
        self.verbose = verbose
        self.history: list[ToolExecution] = []

    def log(self, execution: ToolExecution) -> None:
        """Registra un intento en el historial."""
        self.history.append(execution)
        if self.verbose:
            print(execution.to_report())

    def print_history(self, only_failures: bool = False) -> None:
        """Imprime la tabla de intentos detallando el número de reintento."""
        if not self.history:
            print("\n[HISTORIAL] No hay llamadas registradas en la sesión.")
            return

        total = len(self.history)
        failures_count = sum(1 for e in self.history if not e.ok)
        successes_count = total - failures_count

        records = (
            [e for e in self.history if not e.ok] if only_failures else self.history
        )

        if only_failures and not records:
            print(
                f"\n[HISTORIAL] ✅ Sin errores: todos los intentos ({total}) fueron exitosos."
            )
            return

        if only_failures:
            error_rate = (failures_count / total) * 100
            header_title = (
                f"HISTORIAL DE FALLOS [FILTRO: SOLO ERRORES] "
                f"({len(records)} de {total} intentos | Tasa de fallo: {error_rate:.1f}%)"
            )
        else:
            header_title = (
                f"HISTORIAL GLOBAL DE INTENTOS "
                f"(Total: {total} | Éxitos: {successes_count} | Fallos: {failures_count})"
            )

        formatted_rows: list[str] = []
        for i, exec_item in enumerate(records, start=1):
            status = "OK" if exec_item.ok else "ERROR"
            time_str = f"{exec_item.elapsed_seconds:.3f}s"
            hora_str = exec_item.timestamp.strftime("%H:%M:%S")
            attempt_str = f"{exec_item.attempt}/{exec_item.max_retries}"

            if (
                only_failures
                and isinstance(exec_item.output, dict)
                and "message" in exec_item.output
            ):
                detail = f"[{exec_item.output.get('error_type', 'Error')}] {exec_item.output['message']}"
            else:
                detail = json.dumps(exec_item.arguments, ensure_ascii=False)

            formatted_rows.append(
                f"{i:<4} {hora_str:<9} {exec_item.name:<20} {attempt_str:<9} {status:<8} {time_str:<10} {detail}"
            )

        table_width = max(
            [len(header_title), 85] + [len(row) for row in formatted_rows]
        )
        sep_main = "=" * table_width
        sep_sub = "-" * table_width

        print("\n" + sep_main)
        print(header_title)
        print(sep_main)
        print(
            f"{'#':<4} {'HORA':<9} {'HERRAMIENTA':<20} {'INTENTO':<9} {'ESTADO':<8} {'TIEMPO':<10} {'ARGS / DETALLE'}"
        )
        print(sep_sub)
        for row in formatted_rows:
            print(row)
        print(sep_main + "\n")
        
    def clear(self) -> None:
            """Limpia el historial de la sesión."""
            self.history.clear()


session_tool_logger = ToolExecutionLogger(verbose=False)

# 1. Herramienta de RAG

## 1.1 Carga de datos

Se usará el fichero PDF local: `data/raw/TENERIFE.pdf`.

El objetivo es que el contenido acabe convertido en objetos `Document`, que tienen dos partes:

- `page_content`: el texto del documento.
- `metadata`: información adicional, como fuente, página, ruta, fecha, categoría, etc.

In [3]:
from pypdf import PdfReader
from langchain_core.documents import Document

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

reader = PdfReader(str(PDF_PATH))

docs = [
    Document(
        page_content=page.extract_text().strip(),
        metadata={
            "source": str(PDF_PATH),
            "page": page_number,
            "page_label": reader.page_labels[page_number],
            "total_pages": len(reader.pages),
        },
    )
    for page_number, page in enumerate(reader.pages)
]

print("\nNúmero de páginas cargadas:", len(docs))
print("Metadata de la primera página:", docs[0].metadata)
print("Primeros caracteres:\n")
print(docs[0].page_content[:512])


Número de páginas cargadas: 25
Metadata de la primera página: {'source': '..\\..\\data\\raw\\TENERIFE.pdf', 'page': 0, 'page_label': '1', 'total_pages': 25}
Primeros caracteres:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler -


## 1.2 *Chunking*

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)

splits = text_splitter.split_documents(docs)

# Añadimos un identificador simple a cada chunk para poder citarlo luego.
for i, doc in enumerate(splits):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(splits))
print("\nEjemplo de chunk:\n")
print(splits[0].page_content)
print("\nMetadata:", splits[0].metadata)

Número de chunks: 47

Ejemplo de chunk:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo

Metadata: {'source': '..\\..\\data\\raw\\TENERIFE.pdf', 'page': 0, 'page_label': '1', 'total_pages': 25, 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}


## 1.3 *Embeddings* y *vector store*

El objetivo del RAG es recuperar información relacionada con las consultas variables del usuario. Para ello, necesitamos comparar el signficado de la pregunta con el significado de nuestros textos (chunks) para recuperar los que sean más parecidos. Esos chunks recuperados serán provistos al LLM como contexto de tal manera que lo integre en su respuesta al usuario y dé como resultado una respuesta relevante.

Para poder comparar significados, cada texto será transformado a embedding y se compararán usando una medida de similaridad.

Se usará la clase `InMemoryVectorStore` para la transformación y almacenamiento de los embeddings en memoria.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(splits)

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 47
Primeros IDs: ['54275063-efab-4c7c-b983-d5cb155d76f8', 'cfee4e5e-8d33-4688-b9f6-9211d2beef3d', '8f0c0f8a-75c7-44cf-a166-257627a7bf75']


## 1.4 Recuperación del RAG

In [6]:
class SearchTenerifeDocsInput(BaseModel):
    query: str = Field(
        description=(
            "Términos o pregunta para buscar en la guía interna de Tenerife. "
            "Ejemplos: 'qué ver en La Laguna', 'dónde comer en el norte guachinche', "
            "'playas en La Orotava', 'miradores en Anaga', 'rutas y accesos al Teide', "
            "'beach clubs en Costa Adeje', 'bocadillos de pollo'."
        )
    )
    k: int = Field(
        default=RAG_K,
        ge=1,
        le=8,
        description="Número de fragmentos relevantes a recuperar (entre 1 y 8).",
    )


@tool(args_schema=SearchTenerifeDocsInput)
def search_tenerife_docs(query: str, k: int = 4) -> list[dict[str, Any]]:
    """OBLIGATORIO: Búsqueda en la guía personal de Tenerife suministrada para el proyecto.
    Contiene recomendaciones exclusivas de un guía local sobre distintos lugares, beach clubs,
    gastronomía, ocio y logística de desplazamientos.
    DEBES invocarla SIEMPRE que el usuario pida recomendaciones, rutas, playas,
    dónde comer o curiosidades de Tenerife antes de responder.
    """
    
    # similarity_search_with_score recibe 'query' y 'k'
    retrieved_docs = vector_store.similarity_search_with_score(query, k=k)

    if not retrieved_docs:
        return [{"message": f"No se encontró información relevante para '{query}'."}]

    results: list[dict[str, Any]] = []
    for doc, score in retrieved_docs:
        results.append(
            {
                "score": round(float(score), 4),
                "source": doc.metadata.get("source_name", "Guía de Tenerife"),
                "chunk_id": doc.metadata.get("chunk_id", "desconocido"),
                "content": doc.page_content.strip(),
            }
        )

    return results

# 2. Herramienta de consulta meteorológica ``get_weather``

In [7]:
class WeatherInput(BaseModel):
    location: str = Field(
        description=(
            "Nombre del municipio o zona de Tenerife a consultar (ej. 'Puerto de la Cruz', "
            "'San Cristóbal de La Laguna', 'Adeje', 'La Orotava', 'Santa Cruz de Tenerife', 'El Teide'). "
            "Si el usuario pregunta por 'allí', 'en esa playa' o hace referencia al lugar "
            "comentado en el turno anterior, deduce el municipio exacto y colócalo aquí."
        )
    )
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Unidad de medida para la temperatura. Por defecto 'celsius'.",
    )

def geocode_location(location: str) -> dict[str, Any]:
    """Obtiene coordenadas para una ubicación usando Open-Meteo Geocoding."""
    response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "es", "format": "json"},
        timeout=8,
    )
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results") or []

    if not results:
        raise ValueError(f"No se encontraron coordenadas para '{location}'.")

    first = results[0]
    return {
        "name": first.get("name"),
        "country": first.get("country"),
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "timezone": first.get("timezone"),
    }


# Variable de apoyo para simular caídas de red transitorias en las pruebas
_simulated_network_failures_remaining = 0

@tool(args_schema=WeatherInput)
def get_weather(
    location: str, units: Literal["celsius", "fahrenheit"] = "celsius"
) -> dict[str, Any]:
    """OBLIGATORIO: Consulta el estado meteorológico actual y temperaturas en tiempo real.
    Debes invocarla SIEMPRE que el usuario pregunte por el tiempo, clima, previsión,
    lluvia, viento o temperatura de cualquier lugar de Tenerife o zona mencionada.
    """

    global _simulated_network_failures_remaining

    # Simulación activa de caída de conexión (dispara el mecanismo de reintentos)
    if _simulated_network_failures_remaining > 0:
        _simulated_network_failures_remaining -= 1
        raise requests.exceptions.ConnectionError(
            "Fallo transitorio simulado: timeout de conexión con la API de Open-Meteo."
        )
    
    # Geocodificación con captura de errores de resolución y red
    try:
        place = geocode_location(location)
    except ValueError as val_err:
        return {
            "status": "error",
            "error_type": "LocationNotFound",
            "message": (
                f"No se encontró la ubicación '{location}'. Pide al usuario que "
                "verifique el nombre o indique una población cercana o más conocida."
            ),
        }
    except requests.exceptions.Timeout:
        return {
            "status": "error",
            "error_type": "TimeoutError",
            "message": "El servicio de geocodificación no respondió a tiempo. Inténtalo de nuevo en unos momentos.",
        }
    except requests.exceptions.RequestException as req_err:
        return {
            "status": "error",
            "error_type": "NetworkError",
            "message": f"Error de comunicación con el servicio de geocodificación: {str(req_err)}",
        }

    # Consulta meteorológica con captura de fallos de API y parsing
    temperature_unit = "fahrenheit" if units == "fahrenheit" else "celsius"

    try:       
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
                "temperature_unit": temperature_unit,
                "timezone": "auto",
            },
            timeout=8,
        )
        response.raise_for_status()
        payload = response.json()

        current = payload.get("current") or {}
        units_payload = payload.get("current_units") or {}

        return {
            "status": "success",
            "location": {
                "name": place["name"],
                "country": place.get("country"),
                "latitude": place["latitude"],
                "longitude": place["longitude"],
            },
            "observed_at": current.get("time"),
            "temperature": current.get("temperature_2m"),
            "temperature_unit": units_payload.get("temperature_2m", "°C"),
            "relative_humidity": current.get("relative_humidity_2m"),
            "relative_humidity_unit": units_payload.get("relative_humidity_2m", "%"),
            "wind_speed": current.get("wind_speed_10m"),
            "wind_speed_unit": units_payload.get("wind_speed_10m", "km/h"),
            "provider": "Open-Meteo",
        }

    except requests.exceptions.Timeout:
        return {
            "status": "error",
            "error_type": "TimeoutError",
            "message": f"Tiempo de espera agotado al obtener el pronóstico para {place['name']}.",
        }
    except requests.exceptions.RequestException as req_err:
        return {
            "status": "error",
            "error_type": "WeatherApiError",
            "message": f"Error al consultar la previsión en Open-Meteo: {str(req_err)}",
        }
    except Exception as exc:
        return {
            "status": "error",
            "error_type": "UnexpectedError",
            "message": f"Error inesperado al procesar los datos del clima: {str(exc)}",
        }

# 3. Configuración del modelo

Se implementa un recortador de la ventana de contexto ajustado a una constante prefijada, MAX_HISTORY_TOKENS.

En cuanto a la memoria y la ventana de contexto hay que destacar lo siguiente. `MemorySaver` guarda todo el histórico del hilo, mientras que `trim_messages` actúa como una ventana deslizante dinámica en el nodo de inferencia. La sesión conserva la trazabilidad completa, pero el modelo nunca recibe más tokens de los permitidos.

Al fijar `start_on="human"`, se previene que la poda empiece en un `ToolMessage` suelto, evitando la ruptura del contrato de mensajes de la API.

Para que el proceso sea 100% resiliente a interrupciones se filtran mensajes huérfanos. Si por cualquier corte existe un tool_call sin su ToolMessage, o un ToolMessage sin su llamada, se limpia antes de enviar el prompt con la función `sanitize_message_history`.

In [8]:
# Configuración de límites de tokens
MAX_OUTPUT_TOKENS = 1000
MAX_HISTORY_TOKENS = 2500

tools = [search_tenerife_docs, get_weather]
tools_by_name = {t.name: t for t in tools}

llm = ChatOpenAI(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_tokens=MAX_OUTPUT_TOKENS,
    streaming=True,
)
llm_with_tools = llm.bind_tools(tools, tool_choice="auto")

# Selección del prompt de sistema
ruta_prompt = Path("../../resources/prompt_guia_turistico_tenerife.txt")

with open(ruta_prompt, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

message_trimmer = trim_messages(
    max_tokens=MAX_HISTORY_TOKENS,
    strategy="last",
    token_counter=llm,
    start_on="human",
    include_system=False,  # El System Prompt se inyecta siempre fresco
)

def get_token_count(messages: list[BaseMessage]) -> int:
    """Calcula el número exacto de tokens usando el tokenizador del modelo."""
    return llm.get_num_tokens_from_messages(messages)

def sanitize_message_history(messages: list[BaseMessage]) -> list[BaseMessage]:
    """Elimina llamadas a herramientas huérfanas (sin ToolMessage asociado)

    o ToolMessages descolgados para garantizar el contrato estricto de OpenAI.
    """
    if not messages:
        return []

    # Recolectamos todos los IDs respondidos por ToolMessages
    responded_tool_ids = {
        m.tool_call_id for m in messages if isinstance(m, ToolMessage)
    }

    sanitized: list[BaseMessage] = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            # Filtramos solo las tool_calls que realmente tienen un ToolMessage en la lista
            valid_calls = [
                tc for tc in msg.tool_calls if tc["id"] in responded_tool_ids
            ]
            if valid_calls:
                # Conservamos el mensaje con las llamadas validadas
                sanitized.append(
                    AIMessage(
                        content=msg.content,
                        tool_calls=valid_calls,
                        id=msg.id,
                    )
                )
            elif msg.content:
                # Si tenía texto además del tool_call, lo conservamos como texto puro
                sanitized.append(AIMessage(content=msg.content, id=msg.id))
            # Si no tenía contenido y sus tool_calls quedaron huérfanas, se descarta
        else:
            sanitized.append(msg)

    # Garantizar que el último mensaje no sea un AIMessage con tool_calls si no hay ToolMessage
    while sanitized and isinstance(sanitized[-1], AIMessage) and sanitized[-1].tool_calls:
        sanitized.pop()

    return sanitized

def call_model(state: MessagesState):
    """Nodo del modelo con sanitización de estado, control de tokens e inferencia."""
    raw_messages = state["messages"]

    # 1. Poda de tokens del historial acumulado
    trimmed_messages = message_trimmer.invoke(raw_messages)

    # 2. Saneamiento defensivo ante estados incompletos o cortes bruscos
    safe_messages = sanitize_message_history(trimmed_messages)

    tokens_before = get_token_count(raw_messages)
    tokens_after = get_token_count(safe_messages)

    if len(safe_messages) < len(raw_messages):
        print(
            f"[TOKEN CONTROL] Historial ajustado: {len(raw_messages)} -> {len(safe_messages)} msgs | "
            f"Tokens: {tokens_before} -> {tokens_after} (Límite: {MAX_HISTORY_TOKENS})"
        )

    # 3. Ensamblado del prompt final con System Prompt
    full_prompt = [SystemMessage(content=SYSTEM_PROMPT)] + safe_messages

    # Llamada al LLM
    response = llm_with_tools.invoke(full_prompt)

    return {"messages": [response]}

def audited_tools_node(state: MessagesState, config: RunnableConfig | None = None):
    """Nodo que ejecuta herramientas con soporte de reintentos configurables y logging."""
    configurable = (config or {}).get("configurable", {})
    # Parámetros configurables:
    max_retries: int = configurable.get("max_tool_retries", MAX_TOOL_RETRIES)
    retry_delay: float = configurable.get("tool_retry_delay", 0.5)

    last_message = state["messages"][-1]
    outputs: list[ToolMessage] = []

    for tool_call in last_message.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        call_id = tool_call["id"]
        tool_instance = tools_by_name.get(name)

        final_tool_content = ""
        success = False

        for attempt in range(1, max_retries + 1):
            start_time = time.perf_counter()

            try:
                if not tool_instance:
                    raise ValueError(f"Herramienta no registrada: '{name}'")

                result = tool_instance.invoke(args)
                elapsed = time.perf_counter() - start_time

                # Registro del intento exitoso
                execution = ToolExecution(
                    name=name,
                    arguments=args,
                    ok=True,
                    output=result,
                    elapsed_seconds=elapsed,
                    attempt=attempt,
                    max_retries=max_retries,
                )
                session_tool_logger.log(execution)

                # Serialización estructurada para permitir el parseo de citas en streaming
                if isinstance(result, (list, dict)):
                    final_tool_content = json.dumps(result, ensure_ascii=False)
                else:
                    final_tool_content = str(result)
                
                success = True
                break  # Sale del bucle de reintentos tras el éxito

            except Exception as exc:
                elapsed = time.perf_counter() - start_time

                # Registro de cada fallo individual
                execution = ToolExecution(
                    name=name,
                    arguments=args,
                    ok=False,
                    output={"error_type": type(exc).__name__, "message": str(exc)},
                    elapsed_seconds=elapsed,
                    attempt=attempt,
                    max_retries=max_retries,
                )
                session_tool_logger.log(execution)

                # Si aún quedan reintentos disponibles, esperar antes del siguiente
                if attempt < max_retries:
                    if retry_delay > 0:
                        time.sleep(retry_delay)
                else:
                    # Se agotaron todos los reintentos
                    final_tool_content = (
                        f"Error tras {max_retries} intentos ejecutando '{name}': {str(exc)}"
                    )

        outputs.append(
            ToolMessage(
                content=final_tool_content,
                name=name,
                tool_call_id=call_id,
            )
        )

    return {"messages": outputs}

workflow = StateGraph(state_schema=MessagesState)

workflow.add_node("model", call_model)
workflow.add_node("tools", audited_tools_node)  # Usamos nuestro nodo auditado

workflow.add_edge(START, "model")
workflow.add_conditional_edges("model", tools_condition)
workflow.add_edge("tools", "model")

memory = MemorySaver()
conversation_app = workflow.compile(checkpointer=memory)

## 3.1 Chat *streaming*

Cuando un modelo decide usar herramientas, primero emite chunks de llamada a función (sin texto visible), salta al nodo tools (donde nuestro ToolExecutionLogger imprime su reporte) y vuelve al modelo para generar la redacción final.

Para que los bloques visuales del logger y los tokens de texto no se mezclen en la consola, gestionamos el encabezado de forma dinámica mediante la función `stream_chat`.

In [9]:
def _render_rag_citations(tool_content_raw: str) -> None:
    """Parsea el contenido devuelto por search_tenerife_docs y muestra las citas formateadas."""
    try:
        docs = json.loads(tool_content_raw)
        if not isinstance(docs, list) or not docs:
            return

        print("\n\n📚 [FUENTES CONSULTADAS EN LA GUÍA]")
        print("┌" + "─" * 68)
        for idx, doc in enumerate(docs, start=1):
            source = doc.get("source", "TENERIFE.pdf")
            page = doc.get("page", "N/A")
            chunk_id = doc.get("chunk_id", "desconocido")
            score = doc.get("score")
            score_str = f" | Similitud: {score:.4f}" if score is not None else ""

            # Fragmento de muestra (primeros 90 caracteres para verificar contexto)
            sample_text = doc.get("content", "").replace("\n", " ")[:90]

            print(f"│ [{idx}] Archivo: {source} | Pág: {page} | Chunk: {chunk_id}{score_str}")
            if sample_text:
                print(f"│     Contexto: \"{sample_text}...\"")
        print("└" + "─" * 68)

    except Exception:
        # En caso de que el contenido no sea JSON estándar, se omite el formateo tabular
        pass

def stream_chat(
    app,
    user_input: str,
    config: dict,
    *,
    delay_per_char: float = 0.015,  # Segundos por carácter (0.0 para velocidad nativa)
) -> str:
    """Ejecuta el grafo emitiendo la respuesta con velocidad de visualización configurable
    y mostrando las citas de fuentes si se invocó la búsqueda documental.
    """
    print(f"\nUsuario: {user_input}")

    # Contabilizamos los mensajes previos para aislar lo generado en este turno
    estado_previo = app.get_state(config)
    mensajes_antes = len(estado_previo.values.get("messages", []))

    accumulated_text = ""
    assistant_started = False

    for chunk, metadata in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="messages",
    ):
        node = metadata.get("langgraph_node")

        # Filtramos únicamente los fragmentos de texto del modelo
        if node == "model" and chunk.content:
            if not assistant_started:
                print("\nAsistente: ", end="", flush=True)
                assistant_started = True

            accumulated_text += chunk.content

            # Visualización con cadencia controlada
            if delay_per_char > 0:
                for char in chunk.content:
                    print(char, end="", flush=True)
                    time.sleep(delay_per_char)
            else:
                print(chunk.content, end="", flush=True)

    # Comprobamos si en este turno se ejecutó la herramienta RAG
    estado_posterior = app.get_state(config)
    nuevos_mensajes = estado_posterior.values.get("messages", [])[mensajes_antes:]

    for msg in nuevos_mensajes:
        if isinstance(msg, ToolMessage) and msg.name == "search_tenerife_docs":
            _render_rag_citations(msg.content)

    print("\n" + "─" * 70)
    return accumulated_text

# 4. Testing

## 4.1 Recuperación exitosa tras 2 fallos transitorios

In [10]:
# Simulamos 2 caídas transitorias de conexión
_simulated_network_failures_remaining = 2

config_resiliente = {
    "configurable": {
        "thread_id": "turista_reintentos_demo",
        "max_tool_retries": MAX_TOOL_RETRIES,      # <- Máximo de reintentos configurado
        "tool_retry_delay": 0.2,     # <- Retardo entre intentos (segundos)
    }
}

res = stream_chat(
    conversation_app,
    "¿Qué previsión hay para Santa Cruz de Tenerife?",
    config=config_resiliente,
    delay_per_char=STREAMING_DELAY_PER_CHAR,
)# 


Usuario: ¿Qué previsión hay para Santa Cruz de Tenerife?

Asistente: Mi niño, ahora mismo en Santa Cruz de Tenerife estamos disfrutando de un clima bastante agradable, con una temperatura de unos 23.8°C y un viento suave de 13.4 km/h. La humedad relativa está en torno al 67%, así que el día promete ser bastante llevadero para pasear o pegarse un salto a la playa. Pero si quieres que te diga qué va a pasar en las próximas horas o días, dime y te consulto la previsión completa.
──────────────────────────────────────────────────────────────────────


## 4.2 Agotamiento de reintentos (fallo definitivo)

Configuramos max_tool_retries: 2 pero forzamos 5 fallos, de modo que se supera el límite.

In [11]:
_simulated_network_failures_remaining = 5

config_estricto = {
    "configurable": {
        "thread_id": "turista_fallo_definitivo",
        "max_tool_retries": 2,      # Solo 2 intentos permitidos
        "tool_retry_delay": 0.1,
    }
}

res_fail = stream_chat(
    conversation_app,
    "¿Qué tiempo hará en el macizo de Anaga hoy?",
    config=config_estricto,
    delay_per_char=STREAMING_DELAY_PER_CHAR,
)


Usuario: ¿Qué tiempo hará en el macizo de Anaga hoy?

Asistente: Mi niño, parece que tengo un problemita con la predicción del tiempo en Anaga en este momento. Pero no te preocupes, normalmente en esa zona suele hacer un clima fresco y agradable, ideal para hacer una pateada por sus senderos y disfrutar de sus vistas verdes y frondosas. Si quieres, puedo intentar consultarlo de nuevo o darte una idea general del clima en esa zona. ¿Qué prefieres?
──────────────────────────────────────────────────────────────────────


## 4.3 Diálogo multi-turno

In [14]:
# ---------------------------------------------------------------------------
# 1. Configuración de sesión aislada
# ---------------------------------------------------------------------------
session_id = f"turista_multiturn_{int(time.time())}"
config_demo = {
    "configurable": {
        "thread_id": session_id,
        "max_tool_retries": 3,
        "tool_retry_delay": 0.2,
    }
}

# ---------------------------------------------------------------------------
# 2. Definición de la batería de diálogos
# ---------------------------------------------------------------------------
dialogos = [
    {
        "turno": 1,
        "objetivo": "Búsqueda RAG sobre la guía y citación de fuentes",
        "input": "Hola, voy a pasar unos días en Tenerife y quiero visitar el Lago Martiánez. ¿Qué me puedes comentar de ese lugar? ¿En qué municipio se encuentra?",
    },
    {
        "turno": 2,
        "objetivo": "Resolución de contexto previo ('allí') + llamada a get_weather",
        "input": "Suena bien. ¿Qué tiempo hace hoy allí en esa zona para saber si voy ahora por la tarde?",
    },
    {
        "turno": 3,
        "objetivo": "Síntesis multiturno: cruce de datos de ruta (Turno 1) y clima (Turno 2)",
        "input": "Teniendo en cuenta la temperatura y el viento que me acabas de indicar, ¿es suficiente con llevar un cortavientos y calzado ligero o recomiendas abrigo térmico?",
    },
    {
        "turno": 4,
        "objetivo": "Petición de un detalle concreto por el que se preguntará más tarde pero se habrá olvidado por trimming del contexto",
        "input": "Me apetece comer sushi. ¿Qué restaurantes japoneses me recomiendas en la isla?",
    },
    {
        "turno": 5,
        "objetivo": "Manejo de error controlado en herramienta (ubicación no válida)",
        "input": "Genial. Y para comparar, ¿qué tiempo hace en PuertoDesconocidoXYZ?",
    },
    {
        "turno": 6,
        "objetivo": "Saturación de la ventana de contexto (1)",
        "input": "Me apetece visitar San Cristóbal de La Laguna. ¿Qué me recomiendas en ese lugar?",
    },
    {
        "turno": 7,
        "objetivo": "Saturación de la ventana de contexto (2)",
        "input": "Me apetece visitar dos beach club en Tenerife. ¿Qué sitios conoces?",
    },
    {
        "turno": 8,
        "objetivo": "Saturación de la ventana de contexto (3)",
        "input": "Me apetece visitar Masca. ¿Qué me recomiendas en ese lugar?",
    },
    {
        "turno": 9,
        "objetivo": "Saturación de la ventana de contexto (4)",
        "input": "Me apetece visitar Siam Park esta tarde. ¿Hará buen tiempo en Adeje?",
    },
    {
        "turno": 10,
        "objetivo": "Error de recuperación de detalles previos en la conversación",
        "input": "Anteriormente en esta conversación me recomendaste algún restaurante japonés para comer sushi, pero lo olvidé. ¿Me lo recuerdas?",
    },
]

# Limpiamos el logger de la sesión para aislar las métricas de esta prueba
session_tool_logger.clear()

print(f"======================================================================")
print(f" CONVERSACIÓN MULTI-TURNO")
print(f"======================================================================\n")

# ---------------------------------------------------------------------------
# 3. Bucle de ejecución interactiva con streaming
# ---------------------------------------------------------------------------
for paso in dialogos:
    print(f"\n[{paso['turno']}/{len(dialogos)}] OBJETIVO: {paso['objetivo']}")
    
    respuesta = stream_chat(
        conversation_app,
        paso["input"],
        config=config_demo,
        delay_per_char=STREAMING_DELAY_PER_CHAR,
    )
    
    # Inspección de memoria en el Checkpointer tras cada turno
    estado_actual = conversation_app.get_state(config_demo)
    mensajes_memoria = estado_actual.values.get("messages", [])
    tokens_actuales = get_token_count(mensajes_memoria)
    
    print(f"\n[ESTADO EN MEMORIA] Mensajes: {len(mensajes_memoria)} | Tokens acumulados: ~{tokens_actuales}")


 CONVERSACIÓN MULTI-TURNO


[1/10] OBJETIVO: Búsqueda RAG sobre la guía y citación de fuentes

Usuario: Hola, voy a pasar unos días en Tenerife y quiero visitar el Lago Martiánez. ¿Qué me puedes comentar de ese lugar? ¿En qué municipio se encuentra?

Asistente: El Lago Martiánez es un conjunto de piscinas naturales que se encuentran en la zona de Playa Martiánez, en el municipio de Puerto de la Cruz, en el norte de Tenerife. Es un lugar muy popular para relajarse y disfrutar del agua en un entorno único, diseñado por el famoso artista César Manrique. Además, cerca de allí puedes visitar otros lugares interesantes como la Playa de Los Patos o la Playa del Ancón, aunque algunas de estas playas requieren caminar un poco y tener en cuenta las mareas para acceder mejor.

¿Quieres que te cuente más sobre cómo llegar, qué hacer en la zona o recomendaciones para comer cerca?

📚 [FUENTES CONSULTADAS EN LA GUÍA]
┌────────────────────────────────────────────────────────────────────
│ [1] Archivo:

# 5. Reportes de ejecución de herramientas

In [13]:
# Historial completo de la sesión
session_tool_logger.print_history()

# Filtrando exclusivamente los fallos controlados
session_tool_logger.print_history(only_failures=True)


HISTORIAL GLOBAL DE INTENTOS (Total: 8 | Éxitos: 5 | Fallos: 3)
#    HORA      HERRAMIENTA          INTENTO   ESTADO   TIEMPO     ARGS / DETALLE
---------------------------------------------------------------------------------------------------------
1    11:08:03  search_tenerife_docs 1/3       OK       0.284s     {"query": "Lago Martiánez"}
2    11:08:06  get_weather          1/3       ERROR    0.000s     {"location": "Puerto de la Cruz"}
3    11:08:07  get_weather          2/3       ERROR    0.001s     {"location": "Puerto de la Cruz"}
4    11:08:07  get_weather          3/3       ERROR    0.001s     {"location": "Puerto de la Cruz"}
5    11:08:17  search_tenerife_docs 1/3       OK       0.259s     {"query": "San Cristóbal de La Laguna"}
6    11:08:21  search_tenerife_docs 1/3       OK       0.187s     {"query": "beach clubs en Tenerife"}
7    11:08:25  search_tenerife_docs 1/3       OK       0.192s     {"query": "Masca"}
8    11:08:30  get_weather          1/3       OK       0.604